# 01 · Data Cleaning and Preparation

**Project:** DataCo Supply Chain Analytics
**Dataset:** [DataCo Smart Supply Chain for Big Data Analysis](https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis) (Kaggle), 180,519 order items, January 2015 – January 2018
**Input:** `data/raw/DataCoSupplyChainDataset.csv`

## Purpose
Turn the raw file into clean, documented data that every later analysis relies on.

## Outputs
1. `data/clean/order_items_clean.csv`: one row per order item
2. `data/clean/orders_clean.csv`: one row per order, used for delivery and fraud analysis to avoid double-counting multi-item orders
3. `reports/data_quality_log.md`: every issue found and the action taken

## Contents
- Phase 2 check: validate the metric definitions against the raw data
- Phase 3A: profile and verify
- Phase 3B: drop, rename, fix types and clean text
- Phase 3C: derived columns, order-level table and saved outputs

Metric definitions are in [docs/metric_definitions.md](../docs/metric_definitions.md).

In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/Rawdata_DataCo_Supply_Chain.csv', encoding='latin-1')
df.shape

(180519, 53)

## Phase 2: metric definitions

Before cleaning the data, I confirmed that each metric is defined correctly
and consistently with the project brief. The code below checks:

- that each metric's formula matches its documented definition
- that the units and time windows are consistent
- how edge cases (nulls, zeros, duplicates) are handled

This keeps any cleaning steps from changing what a metric means.


In [2]:
import pandas as pd

df = pd.read_csv('../data/raw/Rawdata_DataCo_Supply_Chain.csv', encoding='latin-1')

# Volume
print('Order items:', len(df))
print('Orders:', df['Order Id'].nunique())
print('Customers:', df['Customer Id'].nunique())
print('Products:', df['Product Card Id'].nunique())

# Financial
revenue = df['Order Item Total'].sum()
profit = df['Order Profit Per Order'].sum()
print(f'Revenue: ${revenue:,.0f}  Profit: ${profit:,.0f}  Margin: {profit/revenue:.1%}')
print(f"Loss item rate: {(df['Order Profit Per Order'] < 0).mean():.1%}")

# Delivery (order level, shipped only)
orders = df.drop_duplicates('Order Id')
shipped = orders[orders['Delivery Status'] != 'Shipping canceled']
print(f"Shipped orders: {len(shipped):,}  Late rate: {shipped['Late_delivery_risk'].mean():.1%}")

# Fraud (order level)
print(f"Fraud orders: {(orders['Order Status'] == 'SUSPECTED_FRAUD').sum():,}")

Order items: 180519
Orders: 65752
Customers: 20652
Products: 118
Revenue: $33,054,402  Profit: $3,966,903  Margin: 12.0%
Loss item rate: 18.7%
Shipped orders: 62,897  Late rate: 57.3%
Fraud orders: 1,488


# Phase 3: Data cleaning
Goal: produce two clean tables (order items and orders) and a data quality log recording every decision.

## 3.1 Setup

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.width', 200, 'display.max_columns', 60, 'display.max_colwidth', 45)
RAW = Path('../data/raw/Rawdata_DataCo_Supply_Chain.csv')
OUT = Path('../data/clean')
OUT.mkdir(parents=True, exist_ok=True)

# Data quality log: one entry per issue found
log = []
def record(issue, columns, rows, action):
    log.append({'issue': issue, 'columns': columns, 'rows_affected': rows, 'action': action})

## 3.2 Load the data
The file is not UTF-8, so it needs `encoding='latin-1'`.

In [4]:
df = pd.read_csv(RAW, encoding='latin-1')
print(f'{df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(3)

180,519 rows x 53 columns


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,Customer Email,Customer Fname,Customer Id,Customer Lname,Customer Password,Customer Segment,Customer State,Customer Street,Customer Zipcode,Department Id,Department Name,Latitude,Longitude,Market,Order City,Order Country,Order Customer Id,order date (DateOrders),Order Id,Order Item Cardprod Id,Order Item Discount,Order Item Discount Rate,Order Item Id,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Order Status,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Cally,20755,Holloway,XXXXXXXXX,Consumer,PR,5365 Noble Nectar Island,725.0,2,Fitness,18.251453,-66.037056,Pacific Asia,Bekasi,Indonesia,20755,1/31/2018 22:56,77202,1360,13.110000,0.04,180517,327.75,0.29,1,327.75,314.640015,91.250000,Southeast Asia,Java Occidental,COMPLETE,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,XXXXXXXXX,Irene,19492,Luna,XXXXXXXXX,Consumer,PR,2679 Rustic Loop,725.0,2,Fitness,18.279451,-66.037064,Pacific Asia,Bikaner,India,19492,1/13/2018 12:27,75939,1360,16.389999,0.05,179254,327.75,-0.80,1,327.75,311.359985,-249.089996,South Asia,Rajastán,PENDING,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,XXXXXXXXX,Gillian,19491,Maldonado,XXXXXXXXX,Consumer,CA,8510 Round Bear Gate,95125.0,2,Fitness,37.292233,-121.881279,Pacific Asia,Bikaner,India,19491,1/13/2018 12:06,75938,1360,18.030001,0.06,179253,327.75,-0.80,1,327.75,309.720001,-247.779999,South Asia,Rajastán,CLOSED,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class


## 3.3 Profile every column
Data type, null count, share of nulls, distinct values and a sample value for each column.

In [5]:
profile = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'nulls': df.isna().sum(),
    'null_%': (df.isna().mean() * 100).round(1),
    'distinct': df.nunique(),
    'sample': df.iloc[0],
})
profile

,dtype,nulls,null_%,distinct,sample
Type,object,0,0.0,4,DEBIT
Days for shipping (real),int64,0,0.0,7,3
Days for shipment (scheduled),int64,0,0.0,4,4
Benefit per order,float64,0,0.0,21998,91.25
Sales per customer,float64,0,0.0,2927,314.640015
Delivery Status,object,0,0.0,4,Advance shipping
Late_delivery_risk,int64,0,0.0,2,0
Category Id,int64,0,0.0,51,73
Category Name,object,0,0.0,50,Sporting Goods
Customer City,object,0,0.0,563,Caguas


**Findings from the profile**
- Product Description is completely empty and Order Zipcode is about 86% empty.
- Customer Email and Customer Password contain one masked value, and Product Status has a single value, so they carry no information.
- Several column pairs have identical distinct counts (e.g. Customer Id and Order Customer Id), which suggests duplicates. These are tested in 3.4 rather than assumed.

## 3.4 Verify suspected duplicate columns
A column is dropped only if its values are identical to another column on every row.

In [6]:
pairs = [('Benefit per order', 'Order Profit Per Order'),
         ('Sales per customer', 'Order Item Total'),
         ('Order Customer Id', 'Customer Id'),
         ('Order Item Cardprod Id', 'Product Card Id'),
         ('Product Category Id', 'Category Id'),
         ('Product Price', 'Order Item Product Price')]

pairs = [(a, b) for a, b in pairs if a in df.columns and b in df.columns]
pd.DataFrame([(a, b, (df[a] == df[b]).all()) for a, b in pairs],
             columns=['column', 'duplicate_of', 'identical'])

,column,duplicate_of,identical
0,Benefit per order,Order Profit Per Order,True
1,Sales per customer,Order Item Total,True
2,Order Customer Id,Customer Id,True
3,Order Item Cardprod Id,Product Card Id,True
4,Product Category Id,Category Id,True
5,Product Price,Order Item Product Price,True


## 3.5 Verify how the financial columns relate
This decides which column is used as revenue and what the profit column really measures.

In [7]:
checks = {
    'Sales = Product Price x Quantity':
        np.isclose(df['Order Item Product Price'] * df['Order Item Quantity'], df['Sales'], atol=0.02).mean(),
    'Order Item Total = Sales - Discount':
        np.isclose(df['Sales'] - df['Order Item Discount'], df['Order Item Total'], atol=0.02).mean(),
    'Profit Ratio = Profit / Order Item Total':
        np.isclose(df['Order Profit Per Order'] / df['Order Item Total'], df['Order Item Profit Ratio'], atol=0.011).mean(),
    'Order Item Id is unique': float(df['Order Item Id'].is_unique),
    'Fully duplicated rows': df.duplicated().sum(),
}
pd.Series(checks, name='share of rows passing').to_frame()

,share of rows passing
Sales = Product Price x Quantity,1.0
Order Item Total = Sales - Discount,1.0
Profit Ratio = Profit / Order Item Total,1.0
Order Item Id is unique,1.0
Fully duplicated rows,0.0


**Decisions**
- Sales is the amount before discount; Order Item Total is after discount, so revenue = Order Item Total.
- Order Profit Per Order is item-level profit despite its name, because it matches each item's profit ratio.
- Order Item Id is unique, so each row is one order item.

## 3.6 Check order-level consistency and status fields
If order details never change within an order, an order-level table can safely be built later.

In [8]:
df.groupby('Order Id')[['Order Status', 'Type', 'Shipping Mode', 'Delivery Status',
                        'Market', 'Order Region', 'Customer Id']].nunique().max()

Order Status       1
Type               1
Shipping Mode      1
Delivery Status    1
Market             1
Order Region       1
Customer Id        1
dtype: int64

In [9]:
pd.crosstab(df['Order Status'], df['Delivery Status'])

Delivery Status,Advance shipping,Late delivery,Shipping canceled,Shipping on time
Order Status,,,,
CANCELED,0,0,3692,0
CLOSED,4809,11109,0,3698
COMPLETE,14136,34199,0,11156
ON_HOLD,2413,5450,0,1941
PAYMENT_REVIEW,420,1082,0,391
PENDING,4857,11712,0,3658
PENDING_PAYMENT,9588,22922,0,7322
PROCESSING,5369,12503,0,4030
SUSPECTED_FRAUD,0,0,4062,0


**Findings**
- Every suspected fraud and cancelled order has Delivery Status "Shipping canceled". These orders never shipped, so they are excluded from the late-delivery analysis. Delivery Status is also excluded from the fraud model, because it is only known after the fraud decision (data leakage).
- Payment type fully determines order status, and all fraud is paid by bank transfer. Real data is rarely this tidy, which suggests the dataset is simulated.

In [10]:
record('Fraud and cancelled orders never shipped', 'Order Status, Delivery Status',
       int(df['Delivery Status'].eq('Shipping canceled').sum()),
       'Excluded from delivery analysis; Delivery Status excluded from fraud model (leakage)')
record('Payment type fully determines order status', 'Type, Order Status', len(df),
       'Kept; noted as synthetic-data limitation')
print(len(log), 'log entries so far')

2 log entries so far
